# 12 - Foundry IQ through Azure API Management

## Required permissions

| Function | Required permission |
|---|---|
| Read existing sources and create the knowledge base | **Search Service Contributor** plus **Search Index Data Reader** (or **Search Index Data Contributor**) on Search |
| Configure the gateway, broker app and shared Web IQ source | **API Management Service Contributor** or equivalent gateway configuration access; permission to manage the broker app and approve its delegated consent; **Search Service Contributor** on Search |
| Retrieve through APIM | Tenant-approved access to the broker app's delegated **`user_impersonation`** scope and access to the indexed SharePoint documents |
| Gateway access to Search | APIM managed identity: **Search Index Data Reader** on Search |

**Resource names:** keep notebook 11's `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This notebook creates only `alice01-apim-kb` and reuses `alice01-kb-blob` / `alice01-kb-sharepoint`. The Web IQ source (`WEBIQ_APIM_SOURCE_NAME`) is shared and read-only in this notebook; it must match the APIM policy. Shared gateway, broker app and services are unchanged. Prefixes prevent naming collisions, not enforce access isolation.

Notebook 11 calls the knowledge base **directly** and stores the Web IQ key in its MCP knowledge source for notebook and portal retrieval. This notebook reuses the Blob and SharePoint sources but selects a separate, keyless Web IQ source. Retrieval goes **through an APIM gateway**, which supplies the key and forwards the signed-in user's Search context using on-behalf-of exchange.

The architecture diagram below separates the caller's token, the user's on-behalf-of token and APIM's managed-identity token, and explains why the broker and SharePoint ingestion apps are different.

What changes for the caller:

| | Notebook 11 (direct) | Notebook 12 (APIM) |
|---|---|---|
| Caller's token | Azure AI Search | the **broker app** (a different app registration) |
| Web IQ key | stored in the Web IQ knowledge source | only in APIM |
| SharePoint permissions | caller's Search token | same user, via the on-behalf-of exchange |
| Caller needs Search RBAC for retrieval | yes | no, only the broker token |

The knowledge base combines notebook 11's Blob and SharePoint sources with the gateway's shared Web IQ source. Source lookup and knowledge-base creation still require direct Search access; only retrieval goes through APIM.

Set the kernel working directory to `c-sharp-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both overriding process settings; set `WORKSHOP_PREFIX` before running (no prefix prompt).


## Architecture: the gateway, tokens and app registrations

<img src="../docs/foundry-iq-apim-architecture.png" alt="Lab 12 architecture: notebook obtains a broker token from Entra, APIM validates it and exchanges it on behalf of the user, then calls Search with its own identity, the user's context and Web IQ key controls; broker and SharePoint ingestion apps have separate roles" width="1200">

[Open the full-size diagram](../docs/foundry-iq-apim-architecture.svg) | [Editable Excalidraw source](../docs/foundry-iq-apim-architecture.excalidraw)

### Follow one retrieval request

1. **The user gets T1, a token for the broker API.** Azure CLI is the client application; the Azure CLI helper (equivalent to `AzureCliCredential`) uses the signed-in user and requests `api://<broker-client-id>/.default`. The broker app exposes the delegated `user_impersonation` scope and pre-authorizes Azure CLI for it. The notebook needs the broker's client ID, not its secret.
2. **The notebook calls APIM with T1.** APIM validates the issuer/tenant, broker audience and delegated scope before processing the request. The broker app identifies the protected API; it is not the gateway's managed identity.
3. **APIM gets T2, a Search token for that same user.** This is the on-behalf-of (OBO) exchange: APIM sends T1 plus the broker app's client ID and secret to Entra. The broker must have the separately consented Azure Cognitive Search delegated `user_impersonation` permission. This downstream Search permission is different from the identically named scope exposed by the broker API. T2 preserves the user; it is not an app-only token.
4. **APIM calls Search with two different identities and the Web IQ controls.** `Authorization: Bearer <T3>` authenticates APIM's managed identity, which needs **Search Index Data Reader**. `x-ms-query-source-authorization: <T2>` supplies the user's context for indexed SharePoint ACL checks. APIM overrides those headers and removes `x-ms-enable-elevated-read`. It also adds the shared Web IQ source's per-source header-name/header-value controls so Search sends `x-apikey` to Web IQ; the caller does not supply that key.
5. **Search retrieves and returns evidence.** The knowledge base references the existing Content Understanding Blob index, existing SharePoint ACL index and separate keyless Web IQ MCP source. Search uses its own managed identity for Foundry model calls. The answer and references return through APIM to the notebook. The indexed SharePoint source reads the index, not the live site; notebook 08's remote SharePoint pattern is not used here.

### Which app or identity are we configuring?

| App / identity | Purpose | Setup and approval |
|---|---|---|
| **Azure CLI client application** | Signs in the user and acquires T1 for the broker API | Existing Microsoft client; pre-authorized by the broker setup script. No new caller app registration is created for this notebook. |
| **APIM broker app registration** (`BROKER_APP_CLIENT_ID`) | Exposes the caller-facing API scope and authenticates the OBO exchange for T2 | `setup-broker-app.ps1`; tenant approval of its delegated Search permission. `broker-client-secret` is stored as a secret APIM named value, never required in the notebook. |
| **APIM system-assigned managed identity** | Authenticates the gateway's backend Search request with T3 | Enable on the existing gateway and separately grant **Search Index Data Reader** on Search. This identity is neither the broker app nor the end user. |
| **Search system-assigned managed identity** | Calls Foundry models; separately, federates to the ingestion app during lab 07 indexing | Model inference permission on Foundry; the ingestion trust/consent belongs to lab 07 setup, not the broker. |
| **SharePoint ingestion app registration** (`SHAREPOINT_APP_CLIENT_ID`) | Authorizes the earlier lab 07 indexing path that populated documents and ACLs | `setup-sharepoint-app.ps1` plus administrator approval/site grant. Search's managed identity is trusted through federation, without a runtime client secret. This app is **not used for APIM's OBO exchange**. |

**Keep the secrets and permissions separate.** APIM holds two different secrets: `broker-client-secret` authenticates the broker to Entra; `webiq-api-key` authorizes Web IQ calls. Neither grants a user access to SharePoint documents. The ingestion app's Graph `Files.Read.All` permission is tenant-wide; adding a selected-site grant does not narrow that separate permission.

**Setup is not the retrieval path.** This notebook still calls Search directly to read source definitions and create `<prefix>-apim-kb`, so the person running those cells needs Search authoring/data roles. The retrieval-only gateway caller uses the broker token instead. APIM's broad service-level data role is not per-user isolation; indexed SharePoint access relies on the preserved user context and ACL configuration. Inspect allowed and denied document cases, not just HTTP 200.

The diagram reflects the policies in [setup-apim-policy.ps1](../scripts/setup-apim-policy.ps1) and the registration in [setup-broker-app.ps1](../scripts/setup-broker-app.ps1). No app registration or gateway changes are performed by this explanation.

## Prerequisites

- Notebook 11 completed with `CLEAN_UP = false`, so `{prefix}-kb-blob` and `{prefix}-kb-sharepoint` exist.
- Existing Search/Foundry resources and an **existing APIM service**, identified by `APIM_RESOURCE_ID` in `.env`. Bicep does not create APIM.
- **Required backend access:** the APIM service's **system-assigned managed identity** must have **Search Index Data Reader on the Azure AI Search service** used by the gateway. Enabling the identity alone is insufficient. **The setup scripts do not grant this role.** Without backend data access, retrieval can return **HTTP 403**, even with a valid broker token and a successful on-behalf-of exchange.
- Completed gateway and broker setup using `scripts\setup-broker-app.ps1` and `scripts\setup-apim-policy.ps1`, including approved delegated consent.
- `BROKER_APP_CLIENT_ID`, `APIM_GATEWAY_URL` and `WEBIQ_APIM_SOURCE_NAME` in `.env`, using the values printed by the setup scripts.
- An active `az login` session in the deployment tenant.

### Grant the required Search role manually

1. On the existing APIM resource, enable **Identity > System assigned** if it is not already enabled.
2. Open the **Azure AI Search resource** targeted by the gateway, then **Access control (IAM) > Add role assignment**.
3. Select **Search Index Data Reader**. For the member, select the APIM service's **system-assigned managed identity**, not the broker app or the signed-in user.
4. Save the assignment at the **Search service scope** and allow time for role propagation before retrying retrieval. Creating this assignment requires role-assignment permissions at that scope.

A role on the APIM resource or Foundry project does not supply this Search data access. The gateway still forwards the caller's Search context separately; this backend grant does not change SharePoint document permissions.

## C# execution

Select the C# kernel and set the working directory to `c-sharp-notebooks`, **not the repository root**. This standalone notebook loads parent `.env`, then `.env.azure`, overriding environment values. It uses .NET built-in HTTP/JSON/process APIs; no console imports, NuGet packages or azd extensions.

The inline CLI helper requests `--resource api://<broker-client-id>`: the equivalent SDK scope is `api://<broker-client-id>/.default`, never a `/user_impersonation` suffix. Configuration defines helpers only: **no authentication, Search requests or retrieval until the next cells**. Dispose HTTP requests/responses per call; dispose the client in the final cell (rerun configuration after disposal).

In an enterprise application with **Assignment required = No**, Users and groups is **not an allowlist**. With Yes, assignments and tenant policies can restrict access. Owning the broker app is not required to retrieve. Azure CLI pre-authorization approves a client/scope, not user assignment; document permissions remain separate.

[Console step-by-step README](../c-sharp-console-apps/12-foundry-iq-apim/README.md) has commands, settings and cost/dependency details.


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Net.Http;
using System.Threading;
using System.Threading.Tasks;
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;

public sealed class ApimSettings(string tenant, string searchEndpoint, string brokerId, string gateway, string prefix,
    string webIqSourceName = "iq-workshop-webiq")
{
    public string Tenant { get; } = NormalizeGuid(tenant);
    public string SearchEndpoint { get; } = searchEndpoint;
    public string BrokerId { get; } = NormalizeGuid(brokerId);
    public string Gateway { get; } = gateway;
    public string Prefix { get; } = prefix;
    public string WebIqSourceName { get; } = webIqSourceName;
    public string BlobSourceName => Prefix + "-kb-blob";
    public string SharePointSourceName => Prefix + "-kb-sharepoint";
    public string BaseName => Prefix + "-kb";
    public string ApimBaseName => Prefix + "-apim-kb";
    public string BrokerResource => "api://" + BrokerId;
    public string BrokerScope => BrokerResource + "/.default";
    static string NormalizeGuid(string value) => Guid.TryParseExact(value, "D", out var id) ? id.ToString("D") : value;

    public static ApimSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". Complete gateway preparation; see .env.example.");
        var settings = new ApimSettings(Required("AZURE_TENANT_ID"), Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'),
            Required("BROKER_APP_CLIENT_ID"), Required("APIM_GATEWAY_URL").TrimEnd('/'), Required("WORKSHOP_PREFIX"),
            Environment.GetEnvironmentVariable("WEBIQ_APIM_SOURCE_NAME")?.Trim() ?? "iq-workshop-webiq");
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Guid.TryParseExact(BrokerId, "D", out _) ||
            !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$") ||
            !Regex.IsMatch(WebIqSourceName, @"^[a-z][a-z0-9-]{1,127}$"))
            throw new InvalidOperationException("Use tenant/broker GUIDs, a 3-20 lowercase letters/digits prefix starting with a letter, and a valid shared source name.");
        bool Root(string value, out Uri? uri) => Uri.TryCreate(value, UriKind.Absolute, out uri) &&
            uri.Scheme == "https" && uri.UserInfo == "" && uri.IsDefaultPort && uri.AbsolutePath == "/" &&
            uri.Query == "" && uri.Fragment == "";
        if (!Root(SearchEndpoint, out var search) ||
            !Regex.IsMatch(search!.Host, @"^[a-z0-9][a-z0-9-]*\.search\.windows\.net$", RegexOptions.IgnoreCase) ||
            !Root(Gateway, out _))
            throw new InvalidOperationException("Use HTTPS Search and APIM service roots without credentials, ports, paths or queries.");
    }

    // Match the preceding labs: deployment dotenv values override .env and process values.
    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim(), value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class ApimTransport(HttpClient client, ApimSettings settings,
    Func<Task<string>>? searchTokenProvider = null, Func<Task<string>>? brokerTokenProvider = null)
{
    public const string ApiVersion = "2026-08-01-preview";
    readonly Func<Task<string>> searchToken = searchTokenProvider ?? (() => AzureCliTokenAsync(settings.Tenant, "https://search.azure.com"));
    readonly Func<Task<string>> brokerToken = brokerTokenProvider ?? (() => AzureCliTokenAsync(settings.Tenant, settings.BrokerResource));

    // Decoding is inspection, not signature/issuer validation. APIM validate-jwt is the enforcement boundary.
    public static JsonObject InspectToken(string token, ApimSettings settings)
    {
        JsonObject claims;
        try
        {
            string[] parts = token.Split('.');
            if (parts.Length != 3) throw new FormatException();
            string encoded = parts[1].Replace('-', '+').Replace('_', '/');
            encoded = encoded.PadRight((encoded.Length + 3) / 4 * 4, '=');
            claims = JsonNode.Parse(Encoding.UTF8.GetString(Convert.FromBase64String(encoded)))?.AsObject()
                ?? throw new JsonException();
            string? audience = claims["aud"]?.GetValue<string>();
            if (audience != settings.BrokerId && audience != settings.BrokerResource)
                throw new InvalidOperationException("Token is not for the broker app. Request api://<broker-client-id>/.default (SDK) or --resource api://<broker-client-id> (CLI).");
            if (claims["tid"]?.GetValue<string>() != settings.Tenant)
                throw new InvalidOperationException("Broker token tenant differs from AZURE_TENANT_ID.");
            if (!(claims["scp"]?.GetValue<string>() ?? "").Split(' ', StringSplitOptions.RemoveEmptyEntries).Contains("user_impersonation"))
                throw new InvalidOperationException("Broker token is missing delegated user_impersonation. Check exposed scope, Azure CLI pre-authorization and consent.");
        }
        catch (Exception error) when (error is FormatException or JsonException)
        {
            throw new InvalidOperationException("Broker token cannot be decoded for inspection; no raw token is printed.");
        }
        foreach (string field in new[] { "aud", "tid", "scp", "preferred_username", "upn", "oid", "azp", "appid" })
            if (claims[field] is JsonNode value) Console.WriteLine(field + ": " + ApimResults.Redact(value.ToString(), token));
        Console.WriteLine("Claims decoded for inspection only; APIM performs cryptographic validation.");
        return claims;
    }

    public async Task InspectTokenAsync() => InspectToken(await brokerToken(), settings);

    public async Task<JsonObject> SendAsync(HttpMethod method, string path, JsonObject? body = null,
        bool gateway = false, bool allowMissing = false, bool diagnostics = false, int timeoutSeconds = 90)
    {
        settings.Validate();
        bool retrieve = path.EndsWith("/retrieve", StringComparison.Ordinal);
        if (retrieve != gateway || (gateway && method != HttpMethod.Post) ||
            (!gateway && !path.StartsWith("knowledgesources/", StringComparison.Ordinal) && !path.StartsWith("knowledgebases/", StringComparison.Ordinal)))
            throw new ArgumentException("Setup uses direct Search; retrieval uses only APIM.");
        if ((method == HttpMethod.Put || method == HttpMethod.Delete) && path != $"knowledgebases/{settings.ApimBaseName}")
            throw new ArgumentException("Only this lab's APIM knowledge base may be changed.");
        string token;
        try
        {
            token = await (gateway ? brokerToken() : searchToken());
            if (gateway) InspectToken(token, settings);
        }
        catch (Exception error) { throw new InvalidOperationException(ApimResults.Redact(error.Message)); }
        string Safe(string value) => ApimResults.Redact(value, token);
        using var request = new HttpRequestMessage(method,
            $"{(gateway ? settings.Gateway + "/iq" : settings.SearchEndpoint)}/{path}?api-version={ApiVersion}");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        if (body is not null) request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(timeoutSeconds));
        try
        {
            using var response = await client.SendAsync(request, timeout.Token);
            string text = await response.Content.ReadAsStringAsync(timeout.Token);
            int status = (int)response.StatusCode;
            Console.WriteLine($"HTTP status: {status}");
            foreach (var header in response.Headers)
                if (header.Key.Contains("request", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Contains("correlation", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Contains("trace", StringComparison.OrdinalIgnoreCase) ||
                    header.Key.Equals("retry-after", StringComparison.OrdinalIgnoreCase))
                    Console.WriteLine(header.Key + ": " + Safe(string.Join(", ", header.Value)));
            if (status == 404 && allowMissing) return new JsonObject();
            if (!response.IsSuccessStatusCode)
            {
                Console.Error.WriteLine($"HTTP {status}; UTC (Date): {response.Headers.Date:R}");
                Console.Error.WriteLine(status switch
                {
                    401 => "Check broker audience/scope, deployment tenant, pre-authorization and downstream Search delegated consent. No automatic re-login.",
                    403 when gateway => "Verify APIM system-assigned managed identity has Search Index Data Reader on Search; then check user document ACLs/network access.",
                    403 => "Setup needs direct user Search authoring/data roles; gateway access does not grant these.",
                    404 => "Complete lab11 with the same prefix and keep its resources; prepare the shared source with setup-apim-policy.ps1.",
                    429 => "Throttling/quota. Inspect Retry-After; no automatic billable retry.",
                    _ => "Inspect request IDs and redacted response details."
                });
                string safe = Safe(text);
                Console.Error.WriteLine("Response body: " + safe[..Math.Min(8000, safe.Length)]);
                throw new HttpRequestException($"HTTP {status}: request failed; inspect redacted diagnostics above.", null, response.StatusCode);
            }
            JsonObject result;
            try
            {
                result = string.IsNullOrWhiteSpace(text) && method != HttpMethod.Post ? new JsonObject()
                    : JsonNode.Parse(text) as JsonObject ?? throw new JsonException();
            }
            catch (JsonException) { throw new InvalidOperationException("Invalid JSON response; no complete result available."); }
            // Never echo a shared source definition: it may contain accidentally stored credentials.
            if (gateway) ApimResults.Print(result, status, diagnostics, token);
            else if (status == 206) throw new InvalidOperationException("Partial setup response; no complete result available.");
            return result;
        }
        catch (HttpRequestException error) { throw new HttpRequestException(Safe(error.Message), null, error.StatusCode); }
        catch (OperationCanceledException) { throw new TimeoutException("Request timed out; no complete result and no automatic retry."); }
    }

    // Uses the existing az login user, like AzureCliCredential; /.default becomes the resource, not /user_impersonation.
    public static ProcessStartInfo TokenStartInfo(string tenant, string resource)
    {
        if (!Guid.TryParseExact(tenant, "D", out _) ||
            (resource != "https://search.azure.com" && (!resource.StartsWith("api://", StringComparison.Ordinal) ||
                !Guid.TryParseExact(resource[6..], "D", out _))))
            throw new ArgumentException("Use a tenant GUID and Search or broker API resource, without a permission suffix.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in new[] { "account", "get-access-token", "--resource", resource, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(argument);
        return start;
    }

    public static async Task<string> AzureCliTokenAsync(string tenant, string resource)
    {
        using var process = Process.Start(TokenStartInfo(tenant, resource)) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync(), errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Check az login --tenant AZURE_TENANT_ID separately.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Check tenant, broker exposed scope/pre-authorization/consent. No automatic re-login.");
        try
        {
            return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
                ? token : throw new JsonException();
        }
        catch (Exception error) when (error is JsonException or InvalidOperationException)
        { throw new InvalidOperationException("Azure CLI returned no usable token."); }
    }

    // Intentional negative probes from Python12; they never print tokens or response bodies.
    public async Task CheckRejectionsAsync()
    {
        settings.Validate();
        string token;
        try { token = await searchToken(); }
        catch (Exception error) { throw new InvalidOperationException(ApimResults.Redact(error.Message)); }
        foreach (string? credential in new string?[] { null, token })
        {
            using var request = new HttpRequestMessage(HttpMethod.Post,
                $"{settings.Gateway}/iq/knowledgebases/{settings.ApimBaseName}/retrieve?api-version={ApiVersion}");
            if (credential is not null) request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", credential);
            request.Content = new StringContent("""{"messages":[{"role":"user","content":[{"type":"text","text":"hello"}]}]}""",
                Encoding.UTF8, "application/json");
            using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
            try
            {
                using var response = await client.SendAsync(request, timeout.Token);
                Console.WriteLine((credential is null ? "No token: " : "Search token (wrong audience): ") + (int)response.StatusCode);
                if ((int)response.StatusCode != 401) throw new InvalidOperationException("Gateway must reject both probes with 401. Check validate-jwt policy.");
            }
            catch (HttpRequestException error) { throw new HttpRequestException(ApimResults.Redact(error.Message, token)); }
            catch (OperationCanceledException) { throw new TimeoutException("Gateway rejection probe timed out."); }
        }
    }
}

public static class ApimResults
{
    static IEnumerable<string> CredentialValues(JsonNode? node, int depth = 0)
    {
        if (depth > 8) yield break;
        if (node is JsonObject obj)
        {
            foreach (var property in obj)
            {
                if (Regex.IsMatch(property.Key, @"(?i)^(?:key|secret|token|x-apikey|(?:webiq[-_]?)?api[-_]?key|(?:broker[-_]?)?client[-_]?secret|access[-_]?token|refresh[-_]?token|password|authorization|.+-header-value)$") &&
                    property.Value is JsonValue value && value.TryGetValue<string>(out string? secret) && !string.IsNullOrEmpty(secret))
                    yield return secret;
                foreach (string nested in CredentialValues(property.Value, depth + 1)) yield return nested;
            }
        }
        else if (node is JsonArray array)
        {
            foreach (JsonNode? item in array)
                foreach (string nested in CredentialValues(item, depth + 1)) yield return nested;
        }
        else if (node is JsonValue value && value.TryGetValue<string>(out string? text))
        {
            JsonNode? parsed = null;
            try { parsed = JsonNode.Parse(text); } catch (JsonException) { }
            foreach (string nested in CredentialValues(parsed, depth + 1)) yield return nested;
        }
    }

    public static string Redact(string text, params string[] secrets)
    {
        JsonNode? parsed = null;
        try { parsed = JsonNode.Parse(text); } catch (JsonException) { }
        var configured = Environment.GetEnvironmentVariables().Cast<System.Collections.DictionaryEntry>()
            .Where(e => Regex.IsMatch(e.Key.ToString()!, @"(?i)(key|secret|token|password)"))
            .Select(e => e.Value?.ToString() ?? "");
        foreach (string secret in secrets.Concat(configured).Concat(CredentialValues(parsed)).Where(s => !string.IsNullOrEmpty(s)).Distinct())
        {
            text = text.Replace(secret, "<redacted>", StringComparison.Ordinal);
            string escaped = secret;
            for (int level = 0; level < 3; level++)
            {
                escaped = JsonSerializer.Serialize(escaped)[1..^1];
                text = text.Replace(escaped, "<redacted>", StringComparison.Ordinal);
            }
            text = text.Replace(Uri.EscapeDataString(secret), "<redacted>", StringComparison.Ordinal);
        }
        // Also protect unconfigured echoed bearer/JWT credentials and secret-valued JSON/header/query fields.
        text = Regex.Replace(text, @"(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+", "Bearer <redacted>");
        text = Regex.Replace(text, @"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]*", "<redacted>");
        text = Regex.Replace(text,
            """(?i)("?(?:x-apikey|api[-_]?key|webiq[-_]?api[-_]?key|broker[-_]?client[-_]?secret|client[-_]?secret|access[-_]?token|refresh[-_]?token|authorization|[a-z0-9-]+-header-value)"?\s*[:=]\s*)("(?:\\.|[^"\\])*"|[^\s,;&}]+)""",
            "$1\"<redacted>\"");
        text = Regex.Replace(text, @"(?i)([?&](?:sig|token|key|code|client_secret|access_token)=)[^&\s""<>]+", "$1<redacted>");
        return text;
    }

    public static void ValidateQuery(string question, string outputMode)
    {
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        if (outputMode is not ("answerSynthesis" or "extractiveData"))
            throw new ArgumentException("Use answerSynthesis or extractiveData.");
    }

    public static JsonObject ObjectData(JsonNode? node)
    {
        if (node is JsonObject obj) return (JsonObject)obj.DeepClone();
        if (node is JsonValue value && value.TryGetValue<string>(out string? text))
        {
            try { return JsonNode.Parse(text) as JsonObject ?? new JsonObject(); }
            catch (JsonException) { }
        }
        return new JsonObject();
    }

    public static string FormatReference(JsonObject reference)
    {
        var data = ObjectData(reference["sourceData"]);
        for (int depth = 0; depth < 3; depth++)
        {
            var content = ObjectData(data["content"]);
            if (content.Count == 0) break;
            data.Remove("content");
            foreach (var property in content) data[property.Key] = property.Value?.DeepClone();
        }
        var source = ObjectData(data["source"]);
        string First(params JsonNode?[] values) => values.Select(v => v?.ToString()).FirstOrDefault(v => !string.IsNullOrEmpty(v)) ?? "";
        string title = First(data["source/file_name"], source["file_name"], data["sourceFile"], data["title"], reference["docKey"]);
        if (title.Length == 0) title = "Source";
        string url = First(data["source/url"], source["url"], data["webUrl"], data["url"]);
        JsonNode? firstNode = data.ContainsKey("page_from") ? data["page_from"] : data["page"];
        JsonNode? lastNode = data.ContainsKey("page_to") ? data["page_to"] : data["pageTo"];
        string pages = "";
        if (firstNode is JsonValue firstValue && firstValue.TryGetValue<int>(out int first) &&
            lastNode is JsonValue lastValue && lastValue.TryGetValue<int>(out int last) && first >= 1 && first <= last)
        {
            pages = first == last ? $"p. {first}" : $"pp. {first}-{last}";
            if (url.Length > 0) url = url.Split('#')[0] + $"#page={first}";
        }
        return $"- [ref_id:{reference["id"]}] {title} {pages} {(url.Length > 0 ? url : "(No source URL returned.)")}";
    }

    public static void Print(JsonObject result, int status, bool diagnostics = false, params string[] secrets)
    {
        secrets = secrets.Concat(CredentialValues(result)).ToArray();
        void Write(string value) => Console.WriteLine(Redact(value, secrets));
        if (diagnostics)
        {
            Console.WriteLine("Raw activity:");
            Write((result["activity"] ?? new JsonArray()).ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
            Console.WriteLine("Raw references:");
            Write((result["references"] ?? new JsonArray()).ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
        }
        if (result["activity"] is not JsonArray activity || activity.Any(s => s is not JsonObject) ||
            result["response"] is not JsonArray messages || messages.Count == 0 ||
            messages.Any(m => m is not JsonObject || m["content"] is not JsonArray content || content.Count == 0 ||
                content.Any(p => p is not JsonObject || p["text"] is not JsonValue text || !text.TryGetValue<string>(out _))) ||
            result["references"] is not null && (result["references"] is not JsonArray refs || refs.Any(r => r is not JsonObject)))
            throw new InvalidOperationException("Malformed retrieval response; no complete result available.");
        Console.WriteLine("\nAnswer or extractive output:");
        foreach (JsonObject message in messages.OfType<JsonObject>())
            foreach (JsonObject part in message["content"]!.AsArray().OfType<JsonObject>()) Write(part["text"]!.ToString());
        Console.WriteLine("\nReferences:");
        foreach (JsonObject reference in (result["references"] as JsonArray ?? new JsonArray()).OfType<JsonObject>())
            Write(FormatReference(reference));
        Console.WriteLine("\nActivity:");
        foreach (JsonObject step in activity.OfType<JsonObject>())
            Write($"- {step["type"]} {step["knowledgeSourceName"]} {step["count"]} {step["error"]}");
        if (status == 206 || activity.OfType<JsonObject>().Any(s =>
            s["error"] is JsonNode error && error.ToJsonString() is not ("null" or "{}" or "[]" or "\"\"" or "false" or "0")))
            throw new InvalidOperationException("Retrieval was partial or a source failed. Inspect activity; --diagnostics shows the same response, without another request.");
        if (result["references"] is not JsonArray references || references.Count == 0)
            Console.WriteLine("No references returned. This alone does not establish an authentication failure.");
    }
}


// Kernel directory is c-sharp-notebooks; dotenv lives at the repository root.
var settings = ApimSettings.Load(Path.GetFullPath(".."));
var client = new HttpClient(new HttpClientHandler { AllowAutoRedirect = false })
{
    Timeout = TimeSpan.FromSeconds(150)
};
// Delegates make offline testing possible; default acquisition uses the existing az login user.
Func<Task<string>> searchTokenProvider = () => ApimTransport.AzureCliTokenAsync(settings.Tenant, "https://search.azure.com");
Func<Task<string>> brokerTokenProvider = () => ApimTransport.AzureCliTokenAsync(settings.Tenant, settings.BrokerResource);
var transport = new ApimTransport(client, settings, () => searchTokenProvider(), () => brokerTokenProvider());
Console.WriteLine("Gateway: " + ApimResults.Redact(settings.Gateway));
Console.WriteLine("Knowledge base: " + ApimResults.Redact(settings.ApimBaseName));


## 1. Get a token for the broker app

The caller requests a token for the **broker app registration**, containing its approved `user_impersonation` permission. With `AzureCliCredential`, request **`api://<broker-client-id>/.default`**: the SDK converts this to the resource `api://<broker-client-id>` for Azure CLI. Passing `/user_impersonation` directly incorrectly includes the permission name in the resource URL and causes **AADSTS500011** before token decoding.

The cell decodes the token to inspect its audience and delegated scope. APIM performs the actual token validation.

### Manual broker-app configuration

In Entra **App registrations**, select the broker app identified by `BROKER_APP_CLIENT_ID`:

1. Under **Expose an API**, set the Application ID URI to `api://<broker-client-id>` and enable a delegated scope named `user_impersonation`.
2. Under **Authorized client applications**, add Azure CLI (`04b07795-8ddb-461a-bbee-02f9e1bf7b46`) and select that exposed scope.
3. Set `api.requestedAccessTokenVersion` to **2** in the app manifest. The app and its enterprise application must exist in the configured deployment tenant.
4. For the later APIM on-behalf-of exchange, add the **Azure Cognitive Search delegated `user_impersonation` permission**, obtain the required consent, and configure the broker credential plus `broker-client-id` and `tenant-id` named values in APIM. `setup-broker-app.ps1` automates these steps.

`/.default` is the SDK token-request syntax, not a new scope to create under Expose an API. Do not create an Application ID URI ending in `/user_impersonation` or follow a login suggestion that appends `/.default` to that permission name. After correcting the configuration cell, rerun it before the token-inspection cell.

In [ ]:
// Decode claims only; actual signature/issuer validation belongs to APIM validate-jwt.
await transport.InspectTokenAsync();

## 2. Check the shared Web IQ source and create the knowledge base

The Blob and SharePoint sources already exist from notebook 11. `setup-apim-policy.ps1` creates the **shared** Web IQ source, named in the APIM policy (`WEBIQ_APIM_SOURCE_NAME`). This notebook only reads that source and does not overwrite it. Unlike notebook 11's direct source, the gateway source stores no key; APIM adds the key as per-source control headers on every request.

These setup calls go directly to Search using the signed-in user's token. Only *retrieval* goes through APIM.

The knowledge base uses `medium` retrieval reasoning effort, allowing more planning effort for questions spanning the internal and web sources. This can increase latency and model usage compared with `low`; it does not guarantee a correct answer. Rerun the knowledge-base creation cell to apply the setting to an existing knowledge base.

In [ ]:
var webIqSource = await transport.SendAsync(HttpMethod.Get, $"knowledgesources/{settings.WebIqSourceName}");
if (webIqSource["kind"]?.ToString() != "mcpServer" ||
    webIqSource["mcpServerParameters"]?["serverURL"]?.ToString() != "https://api.microsoft.ai/v3/mcp")
    throw new InvalidOperationException("Shared source is not the expected Web IQ MCP source. Check WEBIQ_APIM_SOURCE_NAME and gateway preparation.");
// Print only the name, never the shared source definition or any authentication object.
Console.WriteLine("Shared Web IQ source found (read-only): " + ApimResults.Redact(settings.WebIqSourceName));


In [ ]:
var sourceKinds = new Dictionary<string, string>();
foreach (string name in new[] { settings.BlobSourceName, settings.SharePointSourceName })
{
    var source = await transport.SendAsync(HttpMethod.Get, $"knowledgesources/{name}");
    sourceKinds[name] = source["kind"]?.GetValue<string>() is { Length: > 0 } kind ? kind
        : throw new InvalidOperationException("Existing source has no kind. Complete lab11 first.");
}
sourceKinds[settings.WebIqSourceName] = "mcpServer";
// Reuse lab11's model and routing; do not change its KB or any source/index.
var baseKb = await transport.SendAsync(HttpMethod.Get, $"knowledgebases/{settings.BaseName}");
if (baseKb["models"] is not JsonArray models || models.Count == 0 ||
    baseKb["retrievalInstructions"] is not JsonValue instructions || !instructions.TryGetValue<string>(out string routing))
    throw new InvalidOperationException("Lab11 knowledge base needs its model and routing instructions.");
var knowledgeBase = new JsonObject
{
    ["name"] = settings.ApimBaseName,
    ["description"] = "Workshop policies (Blob and SharePoint) plus Web IQ, reached through APIM.",
    ["retrievalInstructions"] = routing.Replace(settings.Prefix + "-kb-webiq", settings.WebIqSourceName, StringComparison.Ordinal),
    ["answerInstructions"] = (baseKb["answerInstructions"]?.ToString() ?? "") + " Answer from the retrieved evidence, not from memory. " +
        "Do not claim external access is unavailable when web results were retrieved. " +
        "If evidence is insufficient, explain what it does not establish; do not offer to fetch sources later " +
        "or ask permission to search, because retrieval has already run. " +
        "Do not infer the latest release from an incomplete list of older versions.",
    ["outputMode"] = "answerSynthesis",
    ["knowledgeSources"] = new JsonArray(sourceKinds.Keys.Select(name => (JsonNode)new JsonObject { ["name"] = name }).ToArray()),
    ["models"] = models.DeepClone(),
    ["retrievalReasoningEffort"] = new JsonObject { ["kind"] = "medium" }
};
await transport.SendAsync(HttpMethod.Put, $"knowledgebases/{settings.ApimBaseName}", knowledgeBase);
Console.WriteLine("Knowledge base ready: " + ApimResults.Redact(settings.ApimBaseName));


## 3. Ask through the gateway

Look at the request headers: only the broker token. There is no Web IQ key, no `x-ms-query-source-authorization` and no Search token. APIM supplies all of them. The URL is the APIM gateway (`/iq/...`), not the Search endpoint.

Start with an internal question. If retrieval returns **403**, verify the required **Search Index Data Reader** assignment for APIM's managed identity in the prerequisites before changing broker-token scopes or Web IQ authentication.

### Optional output and diagnostics

Keep `answerSynthesis` for parity with Python12, or select the supported `extractiveData` mode. Edit each visible question below to ask your own question. All sources include reference source data (including Web IQ URLs), regardless of source selection.

Set `diagnostics = true` to inspect raw activity/references from the **same response**, without an extra request or persistent result cache. Redaction covers keys, tokens and broker secrets in every displayed answer/reference/error/diagnostic. Raw evidence may contain private passages/URLs; do not share outputs.

The common inline formatter accepts sourceData objects/JSON strings and nested JSON content; it retains returned PDF/SharePoint metadata and page ranges without fabricating links/pages. HTTP 206 and source activity errors raise failures rather than passing as complete answers.


In [ ]:
string outputMode = "answerSynthesis"; // Or "extractiveData".
bool diagnostics = false;
ApimResults.ValidateQuery("default question", outputMode);


In [ ]:
{
    string question = "What is our policy for booking international travel?";
    ApimResults.ValidateQuery(question, outputMode);
    // Only the broker bearer token goes to APIM; the gateway supplies OBO, MI and per-source key controls.
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 60,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName, ["kind"] = sourceKinds[settings.BlobSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName, ["kind"] = sourceKinds[settings.SharePointSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName, ["kind"] = "mcpServer",
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            })
    };
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    var result = await transport.SendAsync(HttpMethod.Post, $"knowledgebases/{settings.ApimBaseName}/retrieve",
        payload, gateway: true, diagnostics: diagnostics, timeoutSeconds: 120);
}


The next question needs current public information. The Web IQ key is added by APIM, so the notebook never touches it. Web reference source data is requested and its URL is printed beside the reference ID.

An MCP activity count of 1 means one processed result, not necessarily enough evidence to establish the latest release. Inspect `result["references"]` for the retrieved content. If retrieval succeeded, an answer claiming no external access is a synthesis problem, not evidence of missing authentication.

In [ ]:
{
    string question = "What is the latest stable release of Python, and when was it released?";
    ApimResults.ValidateQuery(question, outputMode);
    // Only the broker bearer token goes to APIM; the gateway supplies OBO, MI and per-source key controls.
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 60,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName, ["kind"] = sourceKinds[settings.BlobSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName, ["kind"] = sourceKinds[settings.SharePointSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName, ["kind"] = "mcpServer",
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            })
    };
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    var result = await transport.SendAsync(HttpMethod.Post, $"knowledgebases/{settings.ApimBaseName}/retrieve",
        payload, gateway: true, diagnostics: diagnostics, timeoutSeconds: 120);
}


Finally a question that needs both: internal policy and the outside world.

In [ ]:
{
    string question = "Does our travel policy meet current public guidance on business travel? Compare them.";
    ApimResults.ValidateQuery(question, outputMode);
    // Only the broker bearer token goes to APIM; the gateway supplies OBO, MI and per-source key controls.
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 90,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName, ["kind"] = sourceKinds[settings.BlobSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName, ["kind"] = sourceKinds[settings.SharePointSourceName],
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName, ["kind"] = "mcpServer",
                ["includeReferences"] = true, ["includeReferenceSourceData"] = true
            })
    };
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    var result = await transport.SendAsync(HttpMethod.Post, $"knowledgebases/{settings.ApimBaseName}/retrieve",
        payload, gateway: true, diagnostics: diagnostics, timeoutSeconds: 150);
}


## 4. Check that the gateway rejects the wrong callers

APIM's `validate-jwt` policy accepts only tokens for the broker app with the `user_impersonation` scope. Two calls that should fail with **401**:

1. no token at all
2. a valid Azure AI Search token (right user, wrong audience)

In [ ]:
// Intentional negative calls only: helper POSTs the canonical "hello" body with no token, then a Search token.
// Both target APIM /iq/.../retrieve, never direct Search retrieval. Require 401; no raw token/body output.
await transport.CheckRejectionsAsync();


## Cleanup

Deletes only this notebook's knowledge base. The shared Web IQ source is retained, along with the gateway, broker app and notebook 11's Blob and SharePoint sources. Notebook 11 has separate opt-in cleanup for its knowledge resources.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    await transport.SendAsync(HttpMethod.Delete, $"knowledgebases/{settings.ApimBaseName}", allowMissing: true);
    Console.WriteLine("Deleted or already absent: " + ApimResults.Redact(settings.ApimBaseName));
}
else Console.WriteLine("Cleanup disabled.");
client.Dispose();


## References

- [Foundry IQ and agentic retrieval](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Per-user access with `x-ms-query-source-authorization`](https://learn.microsoft.com/azure/search/search-document-level-access-overview)
- [Microsoft identity platform: on-behalf-of flow](https://learn.microsoft.com/entra/identity-platform/v2-oauth2-on-behalf-of-flow)
- [APIM `validate-jwt` policy](https://learn.microsoft.com/azure/api-management/validate-jwt-policy)
- [APIM `authentication-managed-identity` policy](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy)